# **1. Load LLMs model via VLLM**

vllm serve "/mnt/e/New folder/models/Qwen2.5-3B-Instruct/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1" \
    --dtype float16 \
    --tensor-parallel-size 1 \
    --port 8000


# **2. Initial model paras**

In [ ]:
import numpy as np
from dataclasses import dataclass, asdict
from abc import abstractmethod
from typing import Optional, Union, List, Dict
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.neighbors import NearestNeighbors

@dataclass
class ModelConfig:
    """
    Lưu trữ toàn bộ cấu hình suy luận (inference config) dùng để gọi LLM
    chạy trên VLLM server.
    
    Class này chủ yếu chuẩn hóa và convert dữ liệu → Python native
    để JSON hoá gửi lên API.
    """
    # --------- BASIC GENERATION ---------
    max_tokens: int = 75
    # Số lượng tokens tối đa mà model được phép sinh

    stop: Optional[Union[str, List[str]]] = None
    # Các chuỗi stop sequence (khi gặp thì dừng sinh thêm)

    stop_token_ids: Optional[List[int]] = None
    # Token ID nào xuất hiện thì dừng sinh

    ignore_eos: bool = False
    # Nếu True → model sẽ bỏ qua EOS token và tiếp tục sinh

    logprobs: Optional[int] = None
    # Số lượng logprobs muốn server trả về cùng response

    echo: bool = False
    # Nếu True → server sẽ trả lại prompt trong phần output

    # --------- SAMPLING ---------
    temperature: float = 0.7
    # Nhiệt độ sampling (cao → random; thấp → deterministic)

    top_k: Optional[int] = None
    # Lấy top-k tokens mỗi bước sinh (nếu None → disable)

    top_p: float = 0.95
    # Nucleus sampling: lấy nhóm token có cumulative probability ≤ top_p

    min_p: float = 0.0
    # Loại token có xác suất < min_p

    # --------- PENALTIES ---------
    frequency_penalty: float = 0.0
    # Phạt dựa trên tần suất xuất hiện token trước đó

    presence_penalty: float = 0.0
    # Phạt nếu token từng xuất hiện (giảm lặp ý)

    repetition_penalty: float = 1.0
    # Phạt mức độ lặp lại token (1.0 = không phạt)

    length_penalty: float = 1.0
    # Điều chỉnh xác suất theo độ dài (ảnh hưởng text dài/ngắn)

    # --------- TOKEN PROCESSING ---------
    skip_special_tokens: Optional[List[int]] = None
    # Danh sách token ID cần bỏ qua khỏi output (VD: <unk>, <pad>)

    add_special_tokens: Optional[List[int]] = None
    # Token ID bắt buộc thêm vào prompt trước khi gửi lên server

    enable_prefix_caching: bool = True
    # Cho phép VLLM caching prefix để tăng tốc nhiều request giống nhau

    def to_payload(self) -> dict:
        """
        Convert toàn bộ config sang dict để gửi lên vLLM server.
        
        - Chuyển numpy → Python native (.item())
        - Bỏ tất cả keys có giá trị None
        """
        d = asdict(self)

        def convert(v):
            if isinstance(v, np.generic):
                # numpy scalar → python scalar
                return v.item()
            if isinstance(v, list):
                # convert từng phần tử nếu có numpy
                return [convert(x) for x in v]
            return v

        # Loại bỏ những field = None
        return {k: convert(v) for k, v in d.items() if v is not None}

# =====================================================================
# Abstract class chung cho bất kỳ backend LLM nào (OpenAI, VLLM, OLLAMA…)
# =====================================================================
class Abstract_OpenAI_Convex_Hallucinate():
    @abstractmethod
    def generate_responses(self, prompt: str, num_responses: int) -> List | Dict:
        """
        Hàm abstract: lớp con phải implement.
        
        Sinh N câu trả lời từ LLM cho 1 prompt.
        Dùng để phục vụ pipeline Convex Hallucination.
        """
        pass

# =====================================================================
# Class thực tế dùng để gọi API /v1/completions của VLLM server
# =====================================================================
class OpenAI_Convex_Hallucinate(Abstract_OpenAI_Convex_Hallucinate):
    """
    Wrapper dùng để:
    - gửi prompt lên VLLM server
    - cấu hình sampling bằng ModelConfig
    - lấy về N responses để phân tích hallucination
    
    Không liên quan OpenAI API thật, chỉ mô phỏng chuẩn OpenAI format.
    """

    def __init__(
            self,
            model_config: ModelConfig,
            api_url='http://localhost:8000/v1/completions',
            model_name='/mnt/e/New folder/models/Qwen2.5-3B-Instruct/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1',
        ):
        """
        Tham số:

        model_config:
            Cấu hình inference (max_tokens, temperature, top_p...)

        api_url:
            URL REST API của VLLM server (mặc định: localhost)

        model_name:
            Đường dẫn model đã load trong VLLM
            (phải trùng tên lúc khởi tạo server)
        """
        self.api_url = api_url
        self.model_name = model_name
        self.model_config = model_config

    def generate_responses(self, prompt: str, num_responses: int = 50) -> List | Dict:
        """
        Gửi 1 prompt → server và yêu cầu sinh N responses.

        Tham số:
            prompt: chuỗi văn bản gửi lên model
            num_responses: sinh bao nhiêu mẫu (sampling N lần)

        Trả về:
            response['choices'] → danh sách các output theo format OpenAI:
            [
                {"text": "...", "logprobs": ..., ...},
                ...
            ]
        """
        import requests

        # Payload theo chuẩn OpenAI API /v1/completions
        data = {
            'model': self.model_name,
            'prompt': prompt,
            'n': num_responses,            # số lần sampling
            **self.model_config.to_payload()  # merge toàn bộ config vào
        }

        # Gửi POST request
        response = requests.post(self.api_url, json=data).json()

        # Trả lỗi rõ ràng nếu server không đúng format
        if 'choices' not in response:
            raise RuntimeError(response)

        return response['choices']

# **3. Generate responses**

In [6]:
modelconfig = ModelConfig(max_tokens=50,
                          stop=['\n'],
                          temperature=1,
                          top_k=50,
                          top_p=0.8,

                          logprobs=7,
                          echo=False)

model = OpenAI_Convex_Hallucinate(model_config=modelconfig)
prompt = '''You are a concise fact-answering assistant.
Your job is to identify exactly **one** sport associated with a given person.

### Output Format (STRICT)
The model MUST respond using the following structure:

<|im_start|>
THE_ONE_SPORT_NAME
<|im_end|>

### Rules:
- Output **MUST** be only one sport name.
- No reasoning.
- No explanations.
- No multiple sports.
- If multiple sports are possible, choose the most widely recognized sport.

### Few-shot Examples:

<|question|> 
Michael Jordan was a world champion in which sport?
</|question|>
<|im_start|>
basketball
<|im_end|>

<|question|>
Manny Pacquiao was a world champion in which sport?
</|question|>
<|im_start|>
boxing
<|im_end|>

<|question|>
Serena Williams was a world champion in which sport?
</|question|>
<|im_start|>
tennis
<|im_end|>

### Now answer this:
<|question|>
Ezzard Charles was a world champion in which sport?
</|question|>
<|im_start|>
'''
responses = model.generate_responses(prompt=prompt, num_responses=50)

# **4. Defining hallucinate pipeline**

In [ ]:
class Abstract_Convex_Hallucinate():
    """
    Abstract class định nghĩa giao diện cho toàn bộ pipeline đo lường Hallucination
    bằng Convex Archetypal Analysis.

    Các hàm con cần override trong class implement:
    """

    @abstractmethod
    def embedding_responses(self):
        """
        Lấy các câu trả lời (responses) từ LLM và chuyển thành vector embedding.
        Thường là vector từ logprobs của token.
        """
        pass

    @abstractmethod
    def project_simplex_rows(self):
        """
        Chiếu từng hàng của ma trận về không gian simplex:
           - mỗi hàng >= 0
           - tổng mỗi hàng = 1
        Đây là bước bắt buộc trong Archetypal Analysis.
        """
        pass

    @abstractmethod
    def Archetypes_Analysis(self) -> Dict:
        """
        Thực hiện Archetypal Analysis:
            - PCA giảm chiều
            - Khởi tạo A, B (weights)
            - SGD để tìm archetypes Z
            - Trả về:
                A, B, Z_pca, X_reduced, PCA_model
        """
        pass

    @abstractmethod
    def global_volumn(self, Z_pca) -> float:
        """
        Tính Global Volume = log(det(ZZᵀ)).
        Volume càng lớn → không gian phân tán → dễ hallucinate.
        """
        pass

    @abstractmethod
    def global_certainty(self, Z_pca) -> float:
        """
        Tính Global Certainty dựa trên khoảng cách giữa các archetypes.
        Khoảng cách nhỏ → certainty cao → ít hallucinate.
        """
        pass

    @abstractmethod
    def global_score(self, global_volumn, global_certainty)-> float:
        """
        Tổng hợp 2 đại lượng toàn cục để đánh giá Hallucination Score.
        """
        pass

    @abstractmethod
    def semantic_consensus(self, X_reduced, 
                           Z_pca, 
                           n_neighbors, 
                           sigma) -> float:
        """
        Tính Semantic Consensus:
            archetype nào được nhiều sample bao quanh → consensus cao.
        """
        pass

    @abstractmethod
    def usage_rarity(self, B, eps) -> float:
        """
        Usage Rarity đo mức "hiếm" của archetype bằng entropy trên B.
        """
        pass

    @abstractmethod
    def distance_from_consensus(self, Z_pca) -> float:
        """
        Khoảng cách của mỗi archetype đến centroid.
        Xa trung tâm → archetype kỳ dị → dễ là hallucination.
        """
        pass

    @abstractmethod
    def local_certainty(self, X_reduced, B, Z_pca, n_neighbors) -> float:
        """
        Gom 3 local metrics:
            SC, UR, DFC
        để đánh giá từng archetype.
        """
        pass

    @abstractmethod
    def pipeline(self, X_reduced, n_neighbors, eps):
        """
        Pipeline chính:
            - Archetypes
            - Global metrics
            - Local metrics
            - Ranking risk
        """
        pass

class Convex_Hallucinate(Abstract_Convex_Hallucinate):
    """
    Class thực thi toàn bộ pipeline tính toán Hallucination Score bằng
    Convex Archetypal Analysis từ responses của LLM.
    """
    def __init__(self,
                 responses: List|Dict,

                 nums_Archetypes: int = 8,
                 normalize_dims: int = 512,
                 max_iter: int = 100,
                 lr: float = 0.05,
                 verbose: bool = False,
                 random_state: int = 24
                 ):
        """
        Initialize the Convex_Hallucinate module.

        Parameters
        ----------
        responses : List | Dict
            Danh sách hoặc dictionary các vector đầu vào (ví dụ: embeddings, 
            semantic vectors, response vectors…). Đây là dữ liệu thô được dùng
            để tìm convex hull và các archetype.

        nums_Archetypes : int, default=8
            Số lượng archetypes cần tìm (K).  
            Đây là số điểm cực đại mô tả "ranh giới ngữ nghĩa" trong không gian
            embedding mà mô hình sẽ suy ra.

        normalize_dims : int, default=512
            Số chiều để chuẩn hóa/ép embedding về cùng kích thước.  
            Nếu responses không đồng nhất về kích thước, module sẽ chiếu hoặc
            padding/truncate về kích thước này.

        max_iter : int, default=100
            Số vòng lặp tối đa của thuật toán tối ưu convex archetypal analysis.  
            Giá trị cao hơn có thể tăng độ chính xác nhưng tốn tính toán hơn.

        lr : float, default=0.05
            Learning rate cho quá trình tối ưu trọng số archetypes.  
            LR lớn giúp hội tụ nhanh nhưng có thể gây dao động; LR nhỏ ổn định 
            hơn nhưng chậm.

        verbose : bool, default=False
            Nếu True, in log chi tiết trong quá trình tối ưu (loss, gradient, 
            trạng thái hội tụ…) giúp debug dễ hơn.

        random_state : int, default=24
            Seed cố định để đảm bảo tái lập kết quả trong quá trình khởi tạo 
            trọng số archetypes và sampling trong tối ưu.

        """
        self.responses = responses
        self.nums_Archetypes = nums_Archetypes
        self.normalize_dims = normalize_dims
        self.max_iter = max_iter
        self.lr = lr
        self.verbose = verbose
        self.random_state = random_state
        if self.responses:
            self.vector_embedding = self.embedding_responses()

    def embedding_responses(self) -> np.ndarray:
        """
        Chuyển mỗi response thành vector embedding bằng cách lấy 
        token_logprobs từ LLM.

        Output:
            padded matrix shape (n_samples, max_seq_len)
        """
        resp = self.responses  # alias

        # Convert single dict → list
        if isinstance(resp, dict):
            responses = [resp]
        elif isinstance(resp, list):
            responses = resp
        else:
            raise TypeError("responses must be a dict or list of dicts")

        vectors = []

        for r in responses:
            lp = r.get("logprobs", {}).get("token_logprobs", [])
            lp = [x for x in lp if x is not None]  # remove None values
            vectors.append(np.array(lp, dtype=float))

        # Empty case
        if not vectors:
            return np.zeros((0, 0))

        # Padding
        max_len = max(v.shape[0] for v in vectors)
        padded = np.stack([
            np.pad(v, (0, max_len - v.shape[0]))
            for v in vectors
        ])

        return padded
    
    def project_simplex_rows(self, X):
        """
        Chiếu mỗi hàng của X vào simplex:
            v >= 0
            sum(v) = 1

        Đây là điều kiện chuẩn cho archetypal decomposition.
        """
        X = X.copy()
        n, d = X.shape
        for i in range(n):
            v = X[i]
            # if already valid simplex (numerical), quick fix
            if np.all(v >= 0) and abs(v.sum() - 1.0) < 1e-8:
                continue
            # projection
            u = np.sort(v)[::-1]
            cssv = np.cumsum(u)
            rho = np.nonzero(u * np.arange(1, d+1) > (cssv - 1))[0]
            if rho.size == 0:
                theta = 0.0
            else:
                rho = rho[-1]
                theta = (cssv[rho] - 1.0) / (rho + 1.0)
            w = np.maximum(v - theta, 0.0)
            # if sum is zero due to negative large values, fallback to uniform
            s = w.sum()
            if s == 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s
            X[i] = w
        return X
    
    def Archetypes_Analysis(self) -> Dict:
        """
        Pipeline Archetypal Analysis gồm:

        1) Chuẩn hóa vector bằng cách trừ mean
        2) Giảm chiều bằng PCA hoặc TruncatedSVD
        3) Khởi tạo A (k,n) và B (n,k) bằng Dirichlet
        4) SGD để tối ưu reconstruction loss:
                X_hat = B @ (A @ X_reduced)
        5) Chiếu A và B về simplex mỗi bước
        6) Trả về:
            - A: archetype → sample weights
            - B: sample → archetype weights
            - Z_pca: archetypes trong PCA space
            - X_reduced: dữ liệu đã giảm chiều
            - pca: model PCA/SVD
        """
        rng = np.random.default_rng(self.random_state)
        n, d = self.vector_embedding.shape
        if n < 2:
            raise ValueError("Need at least 2 samples")
        # center
        Xc = self.vector_embedding - self.vector_embedding.mean(axis=0, keepdims=True)

        # decide pca_dim safely
        pca_dim = int(min(self.normalize_dims, n-1, d))
        if pca_dim <= 0:
            pca_dim = 1

        # compute PCA / SVD
        try:
            if pca_dim < min(n, d) // 2:
                svd = TruncatedSVD(n_components=pca_dim, random_state=self.random_state)
                Xred = svd.fit_transform(Xc)
                pca_obj = svd
            else:
                pca = PCA(n_components=pca_dim, svd_solver='auto', random_state=self.random_state)
                Xred = pca.fit_transform(Xc)
                pca_obj = pca
        except Exception:
            pca_dim = max(1, min(n-1, d))
            pca = PCA(n_components=pca_dim, svd_solver='auto', random_state=self.random_state)
            Xred = pca.fit_transform(Xc)
            pca_obj = pca

        p = Xred.shape[1]

        # init A (k,n) and B (n,k)
        k = self.nums_Archetypes
        A = rng.dirichlet(np.ones(n), size=k)   # (k, n)
        B = rng.dirichlet(np.ones(k), size=n)   # (n, k)

        # SGD
        for it in range(self.max_iter):
            Z = A @ Xred                     # (k, p)
            X_hat = B @ Z                    # (n, p)
            R = X_hat - Xred                 # (n, p)
            loss = 0.5 * np.mean(np.sum(R**2, axis=1))

            grad_B = (R @ Z.T) * (2.0 / n)    # (n, k)
            grad_A = (B.T @ R) @ Xred.T
            grad_A *= (2.0 / n)               # (k, n)

            B = B - self.lr * grad_B
            A = A - self.lr * grad_A

            # project rows onto simplex
            B = np.clip(B, -1e6, None)
            A = np.clip(A, -1e6, None)

            B = self.project_simplex_rows(B)
            A = self.project_simplex_rows(A)

            if self.verbose and (it % max(10, self.max_iter//10) == 0 or it == self.max_iter-1):
                Z = A @ Xred
                X_hat = B @ Z
                R = X_hat - Xred
                loss = 0.5 * np.mean(np.sum(R**2, axis=1))
                print(f"[iter {it+1}/{self.max_iter}] loss={loss:.6e} pca_dim={p}")

        Z = A @ Xred
        return {
            'A': A,                 # (k, n)
            'B': B,                 # (n, k)
            'Z_pca': Z,             # (k, p)
            'X_reduced': Xred,      # (n, p)
            'pca': pca_obj
        }

    def global_volumn(self, Z_pca: np.ndarray) -> float:
        """
        Tính Global Volume = log(det(Z Zᵀ + εI))

        Volume lớn → archetypes trải rộng → semantic space bất ổn.
        """
        k, d = Z_pca.shape
        C = Z_pca @ Z_pca.T
        eps = 1e-6
        volume = np.log(np.linalg.det(C + eps * np.eye(k)))
        return volume
    
    def global_certainty(self, Z_pca):
        """
        Tính Certainty toàn cục:
            = 1 / mean(distance_between_archetypes)

        Khoảng cách càng nhỏ → certainty càng cao.
        """
        from scipy.spatial.distance import pdist, squareform
        
        D = pdist(Z_pca)  # vector khoảng cách
        return 1 / (1e-6 + np.mean(D))

    def global_score(self, global_volumn, global_certainty):
        """
        Kết hợp volume và certainty thành 1 Global Hallucination Score (GHS).

        GHS cao → nguy cơ hallucination cao.
        """
        return -global_volumn + 1/global_certainty

    def semantic_consensus(self, X: np.ndarray, 
                           Z_pca: np.ndarray, 
                           n_neighbors: int = 30, 
                           sigma = None) -> np.ndarray:
        """
        Tính độ đồng thuận semantic (Semantic Consensus – SC):

        Với mỗi archetype z:
            - tìm k láng giềng gần nhất trong X
            - dùng kernel exp(-d^2 / sigma^2)
            - SC = mean(kernel(d))

        SC cao → archetype đại diện cho nhiều mẫu → đáng tin.
        """
        # If X has different feature dim than Z_pca, try to project X via stored PCA
        # If self has 'pca' from Archetypes_Analysis, use it.
        if X.ndim != 2:
            raise ValueError("X must be 2D array")
        if Z_pca.ndim != 2:
            raise ValueError("Z_pca must be 2D array")

        n = X.shape[0]
        # determine whether X is reduced or original by matching dims
        p = Z_pca.shape[1]
        if X.shape[1] != p:
            # try to use stored PCA object
            if hasattr(self, 'pca_obj') and self.pca_obj is not None:
                try:
                    X_red = self.pca_obj.transform(X)
                except Exception:
                    # fallback: raise informative error
                    raise ValueError(f"X features ({X.shape[1]}) != Z_pca features ({p}) and PCA transform failed.")
            else:
                raise ValueError(f"X has {X.shape[1]} features but Z_pca expects {p}. Provide X_reduced or run Archetypes_Analysis first.")
        else:
            X_red = X

        # compute sigma if not provided
        if sigma is None:
            nbr_tmp = NearestNeighbors(n_neighbors=min(5, max(1, n-1))).fit(X_red)
            dists_tmp, _ = nbr_tmp.kneighbors(X_red)
            sigma = np.median(dists_tmp)

        nbrs = NearestNeighbors(n_neighbors=min(n_neighbors, max(1, n))).fit(X_red)

        k = Z_pca.shape[0]
        SC = np.zeros(k)
        for i in range(k):
            z = Z_pca[i:i+1]  # shape (1, p)
            dists, _ = nbrs.kneighbors(z)
            d = dists[0]
            SC[i] = float(np.mean(np.exp(-(d**2) / (sigma**2 + 1e-12))))
        return SC

    def usage_rarity(self, B: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        """
        Usage Rarity (UR) đo độ hiếm của archetype dựa trên entropy của cột B.

        B: (n_samples, k_archetypes)
        UR[k] = entropy(cột k)

        UR cao → archetype ít được dùng → bất thường → dễ là hallucination.
        """
        Bnorm = B / (B.sum(axis=0, keepdims=True) + eps)
        UR = -(Bnorm * np.log(Bnorm + eps)).sum(axis=0)
        return UR

    def distance_from_consensus(self, Z_pca: np.ndarray) -> np.ndarray:
        """
        Tính khoảng cách của mỗi archetype đến centroid.

        DFC lớn → archetype ở ngoại biên → dễ bất thường.
        """
        centroid = Z_pca.mean(axis=0)
        DFC = np.linalg.norm(Z_pca - centroid, axis=1)
        return DFC

    def local_certainty(self, X: np.ndarray, B: np.ndarray, Z_pca: np.ndarray, n_neighbors: int = 10):
        """
        Gom 3 metrics cho từng archetype:
            - SC  (semantic consensus)
            - UR  (usage rarity)
            - DFC (distance from consensus)

        Trả về dict để ranking.
        """
        SC = self.semantic_consensus(X, Z_pca, n_neighbors=n_neighbors, sigma=None)
        UR = self.usage_rarity(B)
        DFC = self.distance_from_consensus(Z_pca)
        return {
            "semantic_consensus": SC,
            "usage_rarity": UR,
            "distance_from_consensus": DFC,
            "Z_pca": Z_pca
        }
    
    def pipeline(self, top_N: int = 6, global_threholds: float = 1.0, n_neighbors: int = 10):
        """
        Pipeline chính:

        1) Chạy Archetypal Analysis → A, B, X_reduced, Z_pca
        2) Tính global metrics:
               GV = global_volume
               GC = global_certainty
               GHS = global hallucination score
        3) So sánh GHS với threshold → cảnh báo hallucination
        4) Tính local metrics: SC, UR, DFC
        5) Kết hợp thành final_score để rank các archetypes
        6) Trả về các archetypes nguy cơ hallucination cao nhất
        """
        print("🔹 Running Archetypal Analysis...")
        result = self.Archetypes_Analysis()

        print("🔹 Computing global metrics...")
        GV = self.global_volumn(Z_pca=result['Z_pca'])
        GC = self.global_certainty(Z_pca=result["Z_pca"])
        GHS = self.global_score(GV, GC)

        print(f"Global Volume: {GV:.4f}")
        print(f"Global Certainty: {GC:.4f}")
        print(f"Global Hallucination Score: {GHS:.4f}")

        global_is_hallucination = GHS > global_threholds  # Ngưỡng tùy chỉnh
        print("⚠️ HIGH RISK HALLUCINATION" if global_is_hallucination else "OK — Stable Semantic Space")

        # --- Tính 3 local metrics ---
        local = self.local_certainty(X=result['X_reduced'], 
                                     B=result['B'], 
                                     Z_pca=result['Z_pca'],
                                     n_neighbors=n_neighbors
                                     )
        
        SC = local["semantic_consensus"]      # (K,)
        UR = local["usage_rarity"]            # (K,)
        DFC = local["distance_from_consensus"]# (K,)

        final_score = (
            0.5 * SC
            + 0.3 * (1 / (1 + UR))
            + 0.2 * (1 / (1 + DFC))
        )


        # ----- 3. RANK -----
        best_idx = np.argsort(final_score)[-top_N:]    # idx tăng → lấy phần cuối
        best_vectors = result['Z_pca'][best_idx]

        return {
            "global_volume": GV,
            "best_vectors": best_vectors,
            "best_idx": best_idx,
            'final_score': final_score
        }


# **5. Executing pipeline**

In [10]:
hallucinate = Convex_Hallucinate(responses, 
                                 nums_Archetypes=9,
                                 normalize_dims=512,
                                 max_iter=90,
                                 lr=0.05,
                                 verbose=True,
                                 random_state=12)

# **6. Result**

In [14]:
final_result = hallucinate.pipeline(top_N=10,
                              global_threholds=1.0,
                              n_neighbors=8)
for i, text in enumerate(responses):
    print(f'{i} --- {text}')
print(responses[final_result['best_idx'][0]])

🔹 Running Archetypal Analysis...
[iter 1/90] loss=3.618599e-01 pca_dim=7
[iter 11/90] loss=3.315740e-01 pca_dim=7
[iter 21/90] loss=2.956742e-01 pca_dim=7
[iter 31/90] loss=2.508395e-01 pca_dim=7
[iter 41/90] loss=2.019093e-01 pca_dim=7
[iter 51/90] loss=1.659229e-01 pca_dim=7
[iter 61/90] loss=1.434172e-01 pca_dim=7
[iter 71/90] loss=1.268642e-01 pca_dim=7
[iter 81/90] loss=1.145928e-01 pca_dim=7
[iter 90/90] loss=1.062277e-01 pca_dim=7
🔹 Computing global metrics...
Global Volume: -84.9972
Global Certainty: 1.1663
Global Hallucination Score: 85.8546
⚠️ HIGH RISK HALLUCINATION
0 --- {'index': 0, 'text': 'boxing', 'logprobs': {'text_offset': [0, 6], 'token_logprobs': [-0.705984890460968, -5.006777428206988e-06], 'tokens': ['boxing', '\n'], 'top_logprobs': [{'boxing': -0.705984890460968, 'box': -0.690359890460968, ' boxing': -5.674735069274902, 'Box': -7.705985069274902, '(box': -7.893485069274902, '.box': -8.112235069274902, 'professional': -9.721610069274902}, {'\n': -5.006777428206988